Local benchmark audit: run the constructed controls first, then a dependent digits workflow. Two tasks train on disjoint shards; their model weights are averaged and fine-tuned on the full training split with a fresh optimizer. These are CPU processes, not Ray/TorchFT or VMs. Helpers are visible in `selective_replication/`. Costs and failure probabilities are editable assumptions.

In [ ]:
import json
import subprocess
import sys
from datetime import datetime
from pathlib import Path

import sklearn
import torch

print("Python:", sys.executable)
print("Torch:", torch.__version__, "scikit-learn:", sklearn.__version__)
# Select Python (ray-dev) as your kernel before running.
ROOT = Path.cwd()
if not (ROOT / "gossip_benchmarks").exists():
    ROOT = ROOT.parent
HELPERS = ROOT / "gossip_benchmarks" / "selective_replication"
assert HELPERS.exists()
RUN = ROOT / "ray_replication_results" / ("audit-" + datetime.now().strftime("%Y%m%d-%H%M%S"))
RUN.mkdir(parents=True, exist_ok=False)

def write_config(name, value):
    path = RUN / (name + ".json")
    with path.open("x") as stream:
        json.dump(value, stream, indent=2)
    return path

def run(script, *args):
    subprocess.run([sys.executable, str(HELPERS / script), *map(str, args)], check=True)

print("Results:", RUN)

In [ ]:
# Mechanism controls: same primary placement, with and without one replica.
# Late loss is deliberately costly: the control checkpoints only at completion.
# Healthy negative control should finish both ways, with more worker time for replication.
CONTROL_CONFIG = {
    "task": {"width": 256, "epochs": 400, "lr": 0.03, "seed": 123, "batch_size": 64},
    "machines": {"safe": {"cost_per_hour": 1.0}, "spot": {"cost_per_hour": 0.25}},
    "primary": "spot", "backup": "safe",
    "profile_repeats": 2, "failure_fraction": 0.65, "deadline_multiplier": 1.25,
    "accounting": {"bill_warm_pool": True, "shared_storage_per_hour": 0.05},
}
CONTROL_REPEATS = 3
control_path = write_config("control-config", CONTROL_CONFIG)
run("controls.py", "--config", control_path, "--folder", RUN / "controls", "--repeats", CONTROL_REPEATS)
control_result = json.loads((RUN / "controls" / "controls.json").read_text())
print(control_result["checks"])
assert control_result["passed"], "Inspect failed/inconclusive controls before interpreting workload results."

In [ ]:
# Workload comparisons retain regular checkpoints for every policy.
# Increase epochs if profiling reports only one or two seconds of training.
COMMON = {"width": 256, "lr": 0.03, "seed": 123, "batch_size": 64}
CONFIG = {
    "machines": {
        "safe": {"cost_per_hour": 1.0},
        "spot_a": {"cost_per_hour": 1.0},
        "spot_b": {"cost_per_hour": 1.0},
    },
    "tasks": {
        "shard-a": {**COMMON, "epochs": 400, "checkpoint_every": 100, "shard": [0, 2]},
        "shard-b": {**COMMON, "epochs": 400, "checkpoint_every": 100, "shard": [1, 2]},
        "finetune": {**COMMON, "epochs": 800, "checkpoint_every": 100,
                     "depends_on": ["shard-a", "shard-b"]},
    },
    "final_tasks": ["finetune"],
    "profile_repeats": 2,
    "failure_machines": ["spot_a", "spot_b"],
    "spot_cost_multipliers": [0.25, 1.0],
    "failure_probabilities": [0.1, 0.5],
    "deadline_multipliers": [1.25, 2.0],
    "fault_time_fractions": [0.3, 0.75],
    "success_target": 0.95,
    # Charge the whole available warm pool, including idle replacement capacity.
    # This optimizes use of a fixed pool, not how many machines to provision.
    # False models first-use billing and excludes pre-use standby costs; label accordingly.
    "accounting": {"bill_warm_pool": True, "shared_storage_per_hour": 0.05},
}
config_path = write_config("workflow-config", CONFIG)
WORKFLOW = RUN / "workflow"
run("run_study.py", "profile", "--config", config_path, "--folder", WORKFLOW)
run("run_study.py", "search", "--folder", WORKFLOW)
searched = json.loads((WORKFLOW / "search.json").read_text())
print("Profiled critical-path lower bound:", searched["parallel_lower_bound_s"], "seconds")
for i, case in enumerate(searched["cases"]):
    print(i, case["name"])
    for name, row in case["policies"].items():
        print(" ", name, "modeled infeasible (still replayed)" if row is None else
              {"cost": row["expected_cost"], "success": row["success_probability"], "plan": row["plan"]})

In [ ]:
# Start with one declared case, then broaden coverage in a new study.
# Every planning scenario is replayed, plus single/correlated held-out faults.
# This cell can take tens of minutes; runtime depends on selected distinct plans.
CASES = [2]
REPEATS = 3
HOLDOUT_FRACTION = 0.55
run("run_study.py", "replay", "--folder", WORKFLOW, "--cases", *CASES,
    "--repeats", REPEATS, "--holdout-fraction", HOLDOUT_FRACTION)

In [ ]:
from IPython.display import Image, display

run("report.py", WORKFLOW)
for filename in ("modeled-cost.png", "measured-comparison.png"):
    path = WORKFLOW / filename
    if path.exists():
        display(Image(filename=str(path)))
print("Share controls/controls.json and workflow/{profile,search,replay}.json plus workflow/summary.md")